# Three-seed stability + selected-budget oracle validation

Run this only after the one-seed smoke test is valid. It trains seeds 0, 1, and 2, then fine-tunes two unseen-budget oracles at 0.40 (cliff candidate) and 0.80 (stable-region control) for 15 epochs. The primary directional test is whether both `G(0.40) > G(0.80)` and `oracle_gap(0.40) > oracle_gap(0.80)` hold in every seed. No predictor regression is fitted in this round. Enable Kaggle GPU, Internet, and attach a secret named `github_token`.

## 1. GPU and secure GitHub clone

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
print("Detected GPUs:", torch.cuda.device_count())
for gpu_index in range(torch.cuda.device_count()):
    print(f"GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)}")
print("CUDA:", torch.version.cuda, "PyTorch:", torch.__version__)

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    github_token = UserSecretsClient().get_secret("github_token")
except Exception as exc:
    raise RuntimeError("Attach a Kaggle secret named 'github_token'.") from exc
assert github_token
REPO_URL = "https://github.com/duyh80456-code/new-pruning.git"
PROJECT_ROOT = Path("/kaggle/working/new-pruning")
askpass = Path("/kaggle/working/.github_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "GITHUB_TOKEN_RUNTIME": github_token})
try:
    command = ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"] if (PROJECT_ROOT / ".git").is_dir() else ["git", "clone", REPO_URL, str(PROJECT_ROOT)]
    subprocess.run(command, env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    git_env.pop("GITHUB_TOKEN_RUNTIME", None)
    github_token = None
assert (PROJECT_ROOT / "configs" / "kaggle_three_seed_oracle.yaml").is_file(), "Push the new validation notebook/config first"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "thop>=0.1.1", "tabulate>=0.9"], check=True)

## 2. Validate the experiment protocol

In [ ]:
from datetime import datetime, timezone
import yaml
BASE_CONFIG = PROJECT_ROOT / "configs" / "kaggle_three_seed_oracle.yaml"
config = yaml.safe_load(BASE_CONFIG.read_text())
assert config["dataset"]["name"].lower() == "cifar100" and config["dataset"]["fake_data"] is False
assert config["model"]["backbone"] == "slimmable_resnet18"
assert config["experiment"]["seeds"] == [0, 1, 2]
assert config["training"]["epochs"] == 20 and config["training"]["width_loss_reduction"] == "mean"
assert config["compression"]["train_widths"] == [0.25, 0.50, 0.75, 1.00]
assert config["compression"]["eval_widths"] == [round(0.25 + 0.05*i, 2) for i in range(16)]
assert config["oracle"]["enabled"] is True
assert config["oracle"]["widths"] == [0.40, 0.80]
assert config["oracle"]["epochs"] == 15
assert config["geometry"]["method"] == "sliced_wasserstein"
assert config["geometry"]["control_methods"] == ["euclidean_mean"]
RUN_NAME = datetime.now(timezone.utc).strftime("kaggle-3seed-oracle-%Y%m%d-%H%M%S")
RUN_DIR = Path("/kaggle/working/new-pruning-outputs") / RUN_NAMEt
config["experiment"]["output_dir"] = str(RUN_DIR)
RESOLVED_CONFIG = Path("/kaggle/working/kaggle_three_seed_oracle_resolved.yaml")
RESOLVED_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(yaml.safe_dump(config, sort_keys=False))
print("Expected wall time: roughly 1.7-2.7 hours on 2xT4, or 2.5-4 hours on 1xT4.")

## 3. CIFAR-100 preflight and experiment launch

In [ ]:
from data import build_loaders
train_loader, val_loader, feature_loader = build_loaders(config, seed=0)
assert len(train_loader.dataset) == 50_000 and len(val_loader.dataset) == 10_000 and len(feature_loader.dataset) == 2_000
del train_loader, val_loader, feature_loader
print("CIFAR-100 preflight: OK")

In [ ]:
import time
from scripts.run_experiment import run
from scripts.run_multi_gpu import run_multi_gpu
started = time.perf_counter()
gpu_count = torch.cuda.device_count()
if gpu_count >= 2:
    print(f"Using two-GPU seed scheduler ({gpu_count} GPUs detected).")
    report_path = run_multi_gpu(RESOLVED_CONFIG, gpu_ids=[0, 1])
else:
    print("Only one GPU detected; using the sequential fallback.")
    report_path = run(RESOLVED_CONFIG)
elapsed_hours = (time.perf_counter() - started) / 3600
print(f"Completed in {elapsed_hours:.2f} hours")
print("Base report:", report_path)

## 4. Validate all three seeds and oracle artifacts

In [ ]:
import json, numpy as np, pandas as pd
for seed in [0, 1, 2]:
    seed_dir = RUN_DIR / f"seed_{seed}"
    assert (seed_dir / "checkpoint.pt").stat().st_size > 0
    training = pd.read_csv(seed_dir / "training_metrics.csv")
    budget = pd.read_csv(seed_dir / "results" / "budget_metrics.csv")
    oracle = pd.read_csv(seed_dir / "results" / "oracle_metrics.csv")
    assert set(training["width"].unique()) == {0.25, 0.50, 0.75, 1.00}
    assert len(budget) == 16 and np.isfinite(budget[["accuracy", "loss", "flops", "params"]]).all().all()
    assert set(oracle["budget"].round(2)) == {0.40, 0.80} and len(oracle) == 2
    assert np.isfinite(oracle[["oracle_accuracy", "oracle_gap", "baseline_to_oracle_wasserstein"]]).all().all()
    oracle_histories = sorted((seed_dir / "results").glob("oracle_training_budget_*.csv"))
    assert len(oracle_histories) == 2
    for history_path in oracle_histories:
        history = pd.read_csv(history_path)
        assert len(history) == 15 and np.isfinite(history[["loss", "accuracy"]]).all().all()
    assert len(list((seed_dir / "features").glob("features_budget_*.pt"))) == 16
    assert len(list((seed_dir / "oracles").glob("oracle_budget_*.pt"))) == 2
print("All three seeds and 6 oracle observations: OK")

## 5. Per-seed and pooled G-vs-P stability

In [ ]:
from scipy.stats import pearsonr, spearmanr
correlation_rows, local_frames, accuracy_frames = [], [], []
for seed in [0, 1, 2]:
    seed_dir = RUN_DIR / f"seed_{seed}"
    local = pd.read_csv(seed_dir / "results" / "local_sensitivity.csv")
    local["seed"] = seed
    local_frames.append(local)
    budget = pd.read_csv(seed_dir / "results" / "budget_metrics.csv")
    budget["seed"] = seed
    accuracy_frames.append(budget)
    pr, sr = pearsonr(local["G_width"], local["P"]), spearmanr(local["G_width"], local["P"])
    summary = json.load(open(seed_dir / "results" / "analysis_summary.json"))
    correlation_rows.append({"seed": seed, "pearson_r": pr.statistic, "pearson_p": pr.pvalue, "spearman_rho": sr.statistic, "spearman_p": sr.pvalue, "pearson_ci_low": summary["sensitivity_accuracy_correlation"]["pearson"]["ci_low"], "pearson_ci_high": summary["sensitivity_accuracy_correlation"]["pearson"]["ci_high"]})
local_all = pd.concat(local_frames, ignore_index=True)
accuracy_all = pd.concat(accuracy_frames, ignore_index=True)
pooled_pr = pearsonr(local_all["G_width"], local_all["P"])
pooled_sr = spearmanr(local_all["G_width"], local_all["P"])
seed_correlations = pd.DataFrame(correlation_rows)
seed_correlations.to_csv(RUN_DIR / "three_seed_correlations.csv", index=False)
display(seed_correlations)
print(f"Pooled Pearson r={pooled_pr.statistic:.4f}, p={pooled_pr.pvalue:.4g}")
print(f"Pooled Spearman rho={pooled_sr.statistic:.4f}, p={pooled_sr.pvalue:.4g}")
print(f"Pearson mean±SD across seeds: {seed_correlations.pearson_r.mean():.4f} ± {seed_correlations.pearson_r.std(ddof=1):.4f}")

In [ ]:
import matplotlib.pyplot as plt
PLOT_DIR = RUN_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for seed, frame in accuracy_all.groupby("seed"):
    axes[0].plot(frame["budget"], frame["accuracy"], marker="o", label=f"seed {seed}")
for seed, frame in local_all.groupby("seed"):
    axes[1].scatter(frame["G_width"], frame["P"], label=f"seed {seed}", alpha=0.8)
axes[0].set(xlabel="Width", ylabel="Accuracy", title="Accuracy stability across seeds")
axes[1].set(xlabel="G(c)", ylabel="P(c)", title="Geometry vs degradation by seed")
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(PLOT_DIR / "three_seed_signal_stability.png", dpi=180); plt.show()

## 6. Assemble the six oracle observations

In [ ]:
central = pd.read_csv(RUN_DIR / "central_analysis_all_seeds.csv")
oracle_data = central.loc[central["oracle_gap_if_available"].notna()].copy()
oracle_data = oracle_data.rename(columns={"oracle_gap_if_available": "oracle_gap", "oracle_accuracy_if_available": "oracle_accuracy"})
assert len(oracle_data) == 6 and set(oracle_data["budget"].round(2)) == {0.40, 0.80}
oracle_table = oracle_data[["seed", "budget", "local_wasserstein_sensitivity", "oracle_gap", "nearest_anchor_distance", "flops_ratio"]].rename(columns={"budget": "width", "local_wasserstein_sensitivity": "pre_oracle_geometry", "nearest_anchor_distance": "coverage"}).sort_values(["seed", "width"]).reset_index(drop=True)
oracle_table["width"] = oracle_table["width"].round(2)
assert np.isfinite(oracle_table[["pre_oracle_geometry", "oracle_gap", "coverage", "flops_ratio"]]).all().all()
oracle_table.to_csv(RUN_DIR / "oracle_040_vs_080_all_seeds.csv", index=False)
display(oracle_table)

## 7. Direct directional hypothesis test

In [ ]:
geometry_wide = oracle_table.pivot(index="seed", columns="width", values="pre_oracle_geometry")
gap_wide = oracle_table.pivot(index="seed", columns="width", values="oracle_gap")
comparison = pd.DataFrame({
    "seed": geometry_wide.index.astype(int),
    "G_0.40": geometry_wide[0.40].to_numpy(),
    "G_0.80": geometry_wide[0.80].to_numpy(),
    "oracle_gap_0.40": gap_wide[0.40].to_numpy(),
    "oracle_gap_0.80": gap_wide[0.80].to_numpy(),
})
comparison["G_0.40_gt_G_0.80"] = comparison["G_0.40"] > comparison["G_0.80"]
comparison["gap_0.40_gt_gap_0.80"] = comparison["oracle_gap_0.40"] > comparison["oracle_gap_0.80"]
comparison["both_hold"] = comparison["G_0.40_gt_G_0.80"] & comparison["gap_0.40_gt_gap_0.80"]
comparison.to_csv(RUN_DIR / "directional_hypothesis_by_seed.csv", index=False)
display(comparison)

## 8. Paired 0.40-versus-0.80 visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for _, row in comparison.iterrows():
    axes[0].plot([0.40, 0.80], [row["G_0.40"], row["G_0.80"]], marker="o", label=f"seed {int(row['seed'])}")
    axes[1].plot([0.40, 0.80], [row["oracle_gap_0.40"], row["oracle_gap_0.80"]], marker="o", label=f"seed {int(row['seed'])}")
axes[0].set(xlabel="Width", ylabel="G(c)", title="Pre-oracle geometry", xticks=[0.40, 0.80])
axes[1].set(xlabel="Width", ylabel="Oracle accuracy gap", title="Oracle recoverability", xticks=[0.40, 0.80])
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(PLOT_DIR / "paired_040_vs_080.png", dpi=180); plt.show()

In [ ]:
print("Definition: G(c) = SW(mu_c, mu_{c+0.05}) / 0.05.")
print("Definition: oracle_gap(c) = oracle_accuracy(c) - shared_accuracy(c).")
print("This round intentionally performs no predictor regression.")

## 9. Decision summary

In [ ]:
r_values = seed_correlations["pearson_r"].to_numpy(float)
stable_dense_signal = np.all(np.sign(r_values) == np.sign(r_values[0])) and np.std(r_values, ddof=1) <= 0.20
oracle_hypothesis_pass = bool(comparison["both_hold"].all())
if oracle_hypothesis_pass:
    decision = "PASS: directional geometry/oracle-gap hypothesis reproduced in all three seeds"
else:
    failed_seeds = comparison.loc[~comparison["both_hold"], "seed"].astype(int).tolist()
    decision = f"NOT CONFIRMED: directional test failed for seed(s) {failed_seeds}"
from IPython.display import Markdown, display
display(Markdown(f"## {decision}"))
print("Per-seed Pearson r:", r_values)
print("Dense G-vs-P stable sign and Pearson SD<=0.20:", stable_dense_signal)
print("G(0.40)>G(0.80) in every seed:", bool(comparison["G_0.40_gt_G_0.80"].all()))
print("oracle_gap(0.40)>oracle_gap(0.80) in every seed:", bool(comparison["gap_0.40_gt_gap_0.80"].all()))
print("Both inequalities hold in every seed:", oracle_hypothesis_pass)

## 10. Persist report and download archive

In [ ]:
report_lines = [
    "# Three-seed + selected-oracle validation", "",
    f"- GPU: {torch.cuda.get_device_name(0)}", "- Dataset/backbone: CIFAR-100 + slimmable ResNet-18",
    "- Shared-model seeds: 0, 1, 2", "- Shared training: 20 epochs, CE + KD, four anchors only",
    "- Oracle widths: 0.40 (cliff candidate), 0.80 (stable control)", "- Oracle fine-tuning: 15 epochs per selected width; six runs total", "",
    "- G(c) = SW(mu_c, mu_{c+0.05}) / 0.05", "- oracle_gap(c) = Acc(M_c*) - Acc(M_c)", "",
    "## Signal stability", "", seed_correlations.to_markdown(index=False), "",
    f"Pooled Pearson: r={pooled_pr.statistic:.6f}, p={pooled_pr.pvalue:.6g}",
    f"Pooled Spearman: rho={pooled_sr.statistic:.6f}, p={pooled_sr.pvalue:.6g}", "",
    "## Six oracle observations", "", oracle_table.to_markdown(index=False), "",
    "## Directional test by seed", "", comparison.to_markdown(index=False), "",
    f"## Decision: {decision}", "",
    f"Both inequalities hold in every seed: {oracle_hypothesis_pass}", "",
    "> This is a targeted three-seed directional replication with only two selected oracle widths. Oracles are independently fine-tuned at a fixed width from each seed's shared checkpoint; no predictor regression is performed in this round.",
]
REPORT = RUN_DIR / "reports" / "kaggle_three_seed_oracle_summary.md"
REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text("\n".join(report_lines) + "\n")
display(Markdown("\n".join(report_lines)))

In [ ]:
import shutil, zipfile
from IPython.display import FileLink
artifact_files = sorted(path for path in RUN_DIR.rglob("*") if path.is_file())
assert len(artifact_files) > 80, f"Unexpectedly few artifacts: {len(artifact_files)}"
manifest = pd.DataFrame({
    "relative_path": [str(path.relative_to(RUN_DIR)) for path in artifact_files],
    "size_bytes": [path.stat().st_size for path in artifact_files],
})
manifest.to_csv(RUN_DIR / "artifact_manifest.csv", index=False)
archive = Path(shutil.make_archive(str(Path("/kaggle/working") / RUN_NAME), "zip", root_dir=RUN_DIR))
assert archive.stat().st_size > 0
with zipfile.ZipFile(archive) as zf:
    archived = set(zf.namelist())
for required in ["artifact_manifest.csv", "central_analysis_all_seeds.csv", "oracle_040_vs_080_all_seeds.csv", "directional_hypothesis_by_seed.csv", "reports/kaggle_three_seed_oracle_summary.md", "seed_0/results/oracle_metrics.csv", "seed_1/results/oracle_metrics.csv", "seed_2/results/oracle_metrics.csv"]:
    assert required in archived, f"Missing from ZIP: {required}"
print(f"Archived {len(archived)} entries; manifest contains {len(manifest)} files.")
print("Run directory:", RUN_DIR)
print("Archive:", archive)
display(FileLink(str(archive)))
print("Use Kaggle Save Version so this multi-hour result persists.")